<a href="https://colab.research.google.com/github/varba187/RAGs-to-Riches/blob/main/code/notebooks/rag_sequence_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install transformers "datasets<3" sentencepiece accelerate faiss-cpu

In [ ]:
import re
import torch
import faiss
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    BartTokenizer,
    BartForConditionalGeneration
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device set to {device}")

torch.manual_seed(0)

# Normalization function

In [ ]:
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

# Retrieval models

In [ ]:
question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
question_encoder = question_encoder.to(device)

In [ ]:
context_tokenizer = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
context_encoder = context_encoder.to(device)

In [ ]:
bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")
bart_model = bart_model.to(device)

# Load NQ

In [ ]:
nq = load_dataset("sentence-transformers/natural-questions", split="train[:500]")
print(nq[0])

In [ ]:
def get_question(example):
  return example["query"]

def get_answer(example):
  answer = example["answer"]
  if isinstance(answer, list):
    answer = answer[0]
  return answer

# Build passage corpus

In [ ]:
def build_passage_corpus(dataset, answer_field_fn):
  passages = []
  for i in range(len(dataset)):
    answer = answer_field_fn(dataset[i])
    if isinstance(answer, list):
      answer = answer[0]
    passages.append(answer)
  passages = list(dict.fromkeys(passages))
  return passages

passages = build_passage_corpus(nq, get_answer)
print(f"Number of passages: {len(passages)}")
print(passages[0])

# Question encoding

In [ ]:
def encode_question(question):
  inputs = question_tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}
  with torch.no_grad():
    outputs = question_encoder(**inputs).pooler_output
  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Passage encoding

In [ ]:
def encode_passage(passage):
  inputs = context_tokenizer(passage, return_tensors="pt", truncation=True, padding=True, max_length=128)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = context_encoder(**inputs).pooler_output

  return outputs.squeeze(0).cpu().numpy().astype(np.float32)

# Token-style retrieval

In [ ]:
def build_sequence_input(question, relevant_passages):
  context = " ".join(relevant_passages)
  return f"question: {question} context: {context}"

# Building FAISS index

In [ ]:
def build_faiss_index(passages):
  passages_embeddings = np.stack([encode_passage(passage) for passage in passages]).astype(np.float32)
  faiss.normalize_L2(passages_embeddings)
  index = faiss.IndexFlatIP(passages_embeddings.shape[1])
  index.add(passages_embeddings)
  return index

# Building retrieval corpus for a dataset

In [ ]:
def retrieval_top_k(question, index, passages, k = 5):
  question_embedding = encode_question(question)
  question_embedding = question_embedding.reshape(1, -1)
  faiss.normalize_L2(question_embedding)
  distances, indices = index.search(question_embedding, k)
  relevant_passages = [passages[i] for i in indices[0]]
  return relevant_passages, distances[0]

In [ ]:
def generate_sequence_answers(question, passages, index, k = 5):
  passages, scores = retrieval_top_k(question, index, passages, k)
  sequence_input = build_sequence_input(question, passages)

  inputs = bart_tokenizer(sequence_input, return_tensors="pt", truncation=True, padding=True, max_length=256)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  with torch.no_grad():
    outputs = bart_model.generate(**inputs, max_new_tokens=32, num_beams=4, no_repeat_ngram_size=2)

  answer = bart_tokenizer.decode(outputs[0], skip_special_tokens=True)
  return answer, passages

# Datasets

In [ ]:
def evaluate_dataset(dataset_name, dataset, question_field_fn, answer_field_fn):
  passages = build_passage_corpus(dataset, answer_field_fn)
  index = build_faiss_index(passages)
  results = []

  for i in range(len(dataset)):
    question = question_field_fn(dataset[i])
    prediction, relevant_passages = generate_sequence_answers(question, index, passages)
    answer = answer_field_fn(dataset[i])

    if isinstance(answer, list):
      answer = answer[0]

    answer_normalized = normalize_text(answer)
    prediction_normalized = normalize_text(prediction)

    em = (int)(answer_normalized==prediction_normalized)
    results.append({
        "dataset": dataset_name,
        "question_number": i,
        "question": question,
        "answer": answer,
        "prediction": prediction,
        "em": em,
        "relevant_passages": relevant_passages
    })

  return pd.DataFrame(results)

# Saving results

In [ ]:
nq_results = evaluate_dataset("NQ", nq, get_question, get_answer)

In [ ]:
all_results = nq_results.copy()
all_results.head()

In [ ]:
summary_df = all_results.groupby("dataset")["em"].mean().reset_index()
summary_df["EM"] = summary_df["em"]*100
summary_df = summary_df.drop(columns=["em"])
summary_df

In [ ]:
all_results.to_csv('rag_sequence_results.csv', index=False)
print("Results saved to rag_sequence_results.csv")